In [1]:
%load_ext autoreload
%autoreload 2

## Initial Preprocessing

In [2]:
import data_preprocessing as d

df = d.load_file()

## Schema and Data Validation 

In [3]:
import data_validations as v

v.validate_against_schema(df)

Since I'm working with a static file I will not be able to compare old vs new - but I will at least save off the old ratios so that I know what I want to compare against eventually

In [4]:
new_stats = v.calculate_psi_statistics(df)
old_stats = v.load_initial_statistics()

In [5]:
for colname in new_stats.keys():
    _, psi = v.population_stability_index(old_stats[colname], new_stats[colname], colname, pregrouped=True)
    if psi > 0.1:
        raise ValueError(f'Population for {colname} has shifted, check before running model')

## Feature Engineering

In [6]:
import feature_engineering as f

In [7]:
#Save some data that I'm likely to need later for streamlit
diag = f.count_all_diag(df)
icd9 = f.load_icd9()
diag = f.join_diag_to_icd9(diag, icd9)
f.stash_data(df, diag)

True

In [8]:
df = f.create_dummies(df)
df = f.parse_weight_to_number(df)

In [9]:
df = f.add_prior_admissions(df)

In [10]:
alive, expired = f.filter_for_still_alive(df)
#Validate that nobody who expired was readmitted later
assert expired[expired['readmitted_dummy']==1].count()['encounter_id'] == 0

In [11]:
alive = f.compress_medicine_columns(alive)

In [12]:
icd9_features = f.icd9_featurization(icd9)
alive = f.add_icd9_features(alive, icd9_features)
alive = f.add_random_feature(alive)

In [13]:
final_feature_df = f.select_final_features(alive)

## Model Training

In [14]:
import model as m

In [15]:
X, y = m.split_X_and_y(final_feature_df)
X_train, X_test, y_train, y_test = m.train_test_split(X,y)

In [17]:
model = m.train_model(X_train, y_train)
output_metrics = m.calculate_metrics(model, X_test, y_test)

In [40]:
champion_model = m.compare_current_to_champion(model, output_metrics, 'precision')

Champion outperforming new version, retaining champion
{'model_hash': '7d3b4ca6998816ab152071a47b434560', 'recall': '0.5491983449702612', 'precision': '0.5776161011763106', 'confusion_matrix': '[[11358, 6212], [6973, 8495]]'}


## Model Calibration